# PyTorch Experience · 01
## `model.to()` — Device · Dtype · Non-blocking

**One glance:** `model.to(device=..., dtype=...)` controls **where the model lives** and **what floating-point type it uses**.

**Docs:** [PyTorch Module.to()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.to) · [Module.eval()](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) · [non_blocking & pin_memory tutorial](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

[← Coding notes](https://lawson-dong.vercel.app/coding)  ·  [DeepSeek Janus reference](https://github.com/deepseek-ai/Janus/blob/main/inference.py)

### 1 · See the syntax

```python
# DeepSeek Janus official style
vl_gpt = vl_gpt.to(torch.bfloat16).cuda().eval()

# Same idea: one .to() call (CUDA with bfloat16 support)
model = model.to(device="cuda", dtype=torch.bfloat16).eval()
```

| Part | Meaning |
|---|---|
| `device="cuda"` | Put model parameters/buffers on the GPU |
| `dtype=torch.bfloat16` | Convert floating-point parameters/buffers to BF16 |
| `.eval()` | Evaluation behavior; **does not** turn off gradients |
| `non_blocking=False` | Default: no request for asynchronous transfer |

### 2 · Run a tiny model (no downloads)

Works in Colab **with or without a GPU**. We use BF16 only when a CUDA GPU reports support.

In [ ]:
import torch
from torch import nn

model = nn.Linear(4, 2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.is_bf16_supported() else torch.float32

model = model.to(device=device, dtype=dtype).eval()
print("Model device:", next(model.parameters()).device)
print("Model dtype: ", next(model.parameters()).dtype)
print("Training mode:", model.training)  # False after eval()

### 3 · The input must match the model

`.to()` **moves the model**, not your input tensor. Make the input use the same device and dtype.

In [ ]:
x = torch.randn(3, 4, device=device, dtype=dtype)
with torch.inference_mode():
    y = model(x)

print("Input:", x.shape, x.device, x.dtype)
print("Output:", y.shape, y.device, y.dtype)  # [3, 2]

### 4 · What about `non_blocking`?

- `non_blocking=False` is the **default**: you can omit it.
- `non_blocking=True` **may** let transfers proceed asynchronously when conditions allow (often with pinned CPU memory). **Not guaranteed**.
- It is about **transfer/synchronization**, not making the model's arithmetic faster.
- **For our Janus `model.to(...).eval()` setup: leave it at the default.**

`model.eval()` and `torch.inference_mode()` are **different**: the former changes model behavior; the latter disables autograd bookkeeping for inference.

**Read:** [Module.to() API](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.to) · [Module.eval() API](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.eval) · [Pinned memory + non-blocking tutorial](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

**Source note:** The Janus line above is from the [DeepSeek repository](https://github.com/deepseek-ai/Janus/blob/main/inference.py); the combined `.to(device=..., dtype=...)` form is our PyTorch rewrite.